In [4]:
# --- Efficient Alignment of Mini-LLMs for Hausa using GRPO and Unsloth ---
# This script aligns the research paper's implementation with the codebase.

# 0. INSTALLATION (Run this in a cell if using Google Colab/Jupyter)
# !pip install unsloth
# !pip install --no-deps trl peft accelerate bitsandbytes
# !pip install xformers

import os
import torch
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from datasets import load_dataset
import re

# 1. Configuration & Layer 1: Base Model (NF4 Quantization)
max_seq_length = 2048
model_name = "unsloth/Llama-3.2-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True, # Layer 1: NF4 Quantization for T4 GPU compatibility
)

# 2. Layer 2: LoRA Adapters (0.75% Parameter Update)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank 16 as per Appendix A & Table 2
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

# 3. Layer 3: GRPO Engine & Reward Hub Logic (Swahili Drift Mitigation)

def hausa_syntax_reward(completions, **kwargs):
    """Rewards specific Hausa orthography and common particles."""
    rewards = []
    # Key Hausa markers: 'na', 'ta', 'yana', 'tana', 'akwai', 'kuma'
    hausa_markers = [r'\bna\b', r'\bta\b', r'\byana\b', r'\btana\b', r'\bkuma\b']
    for content in completions:
        score = sum(1 for m in hausa_markers if re.search(m, content.lower()))
        rewards.append(float(min(score / 3.0, 1.0))) # Normalized reward
    return rewards

def language_uniqueness_reward(completions, **kwargs):
    """
    Implements the recommendation from Section 3.2:
    Penalizes low uniqueness ratios (U < 0.4) to anchor the model in Hausa.
    """
    rewards = []
    for content in completions:
        words = content.lower().split()
        if not words:
            rewards.append(0.0)
            continue
        uniqueness_ratio = len(set(words)) / len(words)
        # Reward high uniqueness, penalize repetition/drift (U < 0.4)
        score = 1.0 if uniqueness_ratio >= 0.4 else -1.0
        rewards.append(score)
    return rewards

def swahili_drift_penalty(completions, **kwargs):
    """Penalizes common Swahili drift tokens observed in initial experiments."""
    penalties = []
    swahili_markers = [r'\bni\b', r'\bna\b', r'\bkatika\b', r'\bkwa\b', r'\bndiyo\b']
    for content in completions:
        score = sum(1 for m in swahili_markers if re.search(m, content.lower()))
        penalties.append(float(-0.5 if score > 2 else 0.0))
    return penalties

# 4. GRPO Training Setup (The Closed-Loop Feedback)
# Updated to match KL Coefficient from Table 2
training_args = GRPOConfig(
    output_dir = "hausa-llama-grpo",
    learning_rate = 5e-6, # 5x10^-6 from Table 2
    beta = 0.01,         # KL Coefficient from Table 2
    lr_scheduler_type = "cosine",
    weight_decay = 0.1,
    max_prompt_length = 256,
    max_completion_length = 512,
    num_generations = 8, # Group Size (G) = 8 from Table 2
    per_device_train_batch_size = 1, # Fits on T4
    gradient_accumulation_steps = 4,
    max_steps = 100,
    save_steps = 50,
    logging_steps = 10,
    report_to = "none",
)

# 5. Layer 4: Inference Filter (Contrastive Search)
def generate_aligned_hausa(prompt):
    FastLanguageModel.for_inference(model)
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

    # Implementing Contrastive Search as per Section 2.4 & Table 2
    outputs = model.generate(
        input_ids = inputs,
        max_new_tokens = 128,
        penalty_alpha = 0.6, # Alpha (a) = 0.6 from Table 2
        top_k = 4,           # Top-k = 4 from Table 2
        use_cache = True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example Execution
if __name__ == "__main__":
    test_prompt = "Yaya ake girka shinkafa?" # How do you cook rice?
    print("Generating aligned Hausa response...")
    print(generate_aligned_hausa(test_prompt))

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 8
Generating aligned Hausa response...
system

Cutting Knowledge Date: December 2023
Today Date: 13 Mar 2026

user

Yaya ake girka shinkafa?assistant

Swali la kina! (Swali la kina ni swali la kina, ambalo linahitaji kuchambua na kufikiria kwa kina.)

Hapa, "Yaya" inaweza kuwa jina la mtu, na "girka" inaweza kuwa jina la mchakato au shughuli, na "shinkafa" inaweza kuwa jina la aina ya mboga au chakula.

Ikiwa ni pamoja na mtu anayehitaji kufanya k
